# Studying the effect of pooling - Design B (crowd-like pool)

Design A (`pooled_design_A_scaled_down_pool.ipynb`) reduces the pooled data to the size of one observer's data. As the same observer answered each triplet three times, design A includes answers of the same observer to the same triplet. Crowd-sourced data do not have such repeats, in crowd-sourced studies no person answers a question twice.

Design B (crowd-like pool) builds pooled data with each triplet answered by the same observer only once. The three answers to a triplet come from three different observers always, to complete the same amount of trials as one observer's data (1800 trials).
This design asks whether the result of design A holds for pooled data built like crowd-sourced data. 

In [1]:
import ast
import os

import numpy as np
import pandas as pd
import sklearn

import cblearn

from load_data import load_all
from dimensionality_estimation import estimate_dimensionality
from io_json import read_json_dataclass, save_json_dataclass
from utils import read_dimensionality
from size_controlled_pooling import REPEAT_TABLE, check_pooling, pool_condition, to_triplets

# seeds the cross-validation splits and the SOE initialisations of every fit;
# draw k of the pooling is seeded with SEED + k
SEED = 0
# number of independent poolings
N_DRAWS = 10
LABEL = 'design-B'
# all fitted runs; read back instead of refitting when it and the jsons exist
RUNS_CSV = '../data/found-dimensionality-embeddings-design-B-draws.csv'
# one row per finished fit while the fits run, so that an interrupted run resumes; deleted once
# RUNS_CSV is written
PARTIAL_CSV = '../data/found-dimensionality-embeddings-design-B-draws.partial.csv'

OBSERVERS = ['O1', 'O2', 'O3', 'O4', 'O5']
TASKS = ['classic', 'odd one out']
IMAGE_SETS = ['action', 'greebles', 'taxonomic']

print('cblearn:', cblearn.__version__)
print('sklearn:', sklearn.__version__)

cblearn: 0.3.0
sklearn: 1.5.2


# Reading data

`load_all(fill_missing_O_by_P=True)` relabels the sessions that O5 did under the pseudonym P1 (1a, 1b, 2a, 2b) as O5, and those of P2 as O6. 
Keeping O1 - O5 gives, for every task and image set, 30 sessions: every observer ran each of the two trial lists (trial sets 1 and 2, 300 triplets each) in three sessions (repeats).

In [2]:
triplet_df = load_all(fill_missing_O_by_P=True)
triplet_df = triplet_df[triplet_df['observer'].isin(OBSERVERS)]
for (obs, task, image_set, triplet_set), sessions in triplet_df.groupby(['observer', 'task', 'image_set',
                                                                         'triplet_set']):
    assert len(sessions) == 3
print(len(triplet_df), 'sessions')

180 sessions


# Design

The goal was to generate pooled data of the size and structure of one observer's data, in which no observer answers the same triplet twice and the five observers are mixed evenly.

A triplet here is one question in one screen layout (`triplet_index`). One observer's data are the 600 triplets of the two trial sets, each answered once in each of three repeats, 600 * 3 = 1800 trials.

For every trial set separately, the 300 triplets are split at random into 10 groups of 30, one per combination of 3 of the 5 observers. `REPEAT_TABLE` (below, observers labelled 0 - 4) gives, for every combination, which of its three observers answers the group in repeat 1, 2 and 3. 
It follows a balanced incomplete block design. 
In every repeat each observer appears exactly twice, so every synthetic repeat has every triplet once and every observer answers a fifth of it. The answers of an observer in repeat $r$ come from its $r$-th session of the trial list, the order of the three sessions is also drawn at random.

Per simulated dataset every observer answers 360 trials in total (30 trials x 2 occurrences per observer x 3 repeats x 2 trial sets) 
Every observer is in 6 combinations, 180 triplets per trial list; every pair of observers shares 3 combinations, 3 x 30 = 90 triplets per trial list; every triplet is answered three times, by three different observers: 3 x 600 = 1800 trials. A triplet that the assigned observer did not answer in its session (a timeout) is kept as missing. 

The list of trials is split into 10 groups at random, for every task, image set and trial set. 
The mapping of O1 - O5 to the labels 0 - 4 and the order of every observer's sessions is also randomly drawn every time. 
The structure of balancing observers and repetitions is fixed in variable `REPEAT_TABLE`. 
The pooling is drawn `N_DRAWS = 10` times; draw $k$ of a task and image set is seeded with `SEED + k` together with the numbers of the task and the image set. All fits use the same seeds for the cross-validation splits and the SOE initialisations, so the draws differ only in the pooling.

In [3]:
pd.DataFrame(REPEAT_TABLE + 0, columns=['repeat 1', 'repeat 2', 'repeat 3'],
             index=[' '.join(map(str, sorted(row))) for row in REPEAT_TABLE]).rename_axis('combination')

,repeat 1,repeat 2,repeat 3
combination,,,
0 1 2,0,1,2
0 1 3,0,1,3
0 1 4,1,0,4
0 2 3,2,0,3
0 2 4,2,4,0
0 3 4,3,4,0
1 2 3,1,3,2
1 2 4,4,2,1
1 3 4,4,3,1


In [4]:
runs_records = []  # one row per draw, task and image set: the data that is fitted
pooled_records = []  # one row per synthetic repeat, for check_pooling
for draw in range(N_DRAWS):
    for task in TASKS:
        for image_set in IMAGE_SETS:
            # one generator per draw and condition, so that a condition's pooling does not depend on
            # which other conditions are run
            rng = np.random.default_rng([SEED + draw, TASKS.index(task), IMAGE_SETS.index(image_set)])
            condition = triplet_df[(triplet_df['task'] == task) & (triplet_df['image_set'] == image_set)]
            repeats = pool_condition(condition, OBSERVERS, rng)
            pooled_records.extend({**r, 'draw': draw, 'task': task, 'image_set': image_set} for r in repeats)
            record = {key: np.concatenate([r[key] for r in repeats])
                      for key in ('responses', 'triplet_index', 'triplet_set', 'source_observer', 'source_session')}
            record['raw_triplets'] = np.vstack([r['raw_triplets'] for r in repeats])
            record['repetition'] = np.concatenate([np.full(len(r['responses']), r['repetition']) for r in repeats])
            runs_records.append({'observer': LABEL, 'draw': draw, 'seed': SEED + draw, 'task': task,
                                 'image_set': image_set, 'n_trials': len(record['responses']), **record,
                                 'embedding_ready_triplets': to_triplets(task, record['raw_triplets'],
                                                                         record['responses'])})

pooled_df = pd.DataFrame(pooled_records)
for draw, draw_df in pooled_df.groupby('draw'):
    check_pooling(draw_df)
runs_df = pd.DataFrame(runs_records)
runs_df['n_triplets'] = runs_df['embedding_ready_triplets'].apply(len)
assert (runs_df['n_triplets'] == runs_df['n_trials'] * np.where(runs_df['task'] == 'classic', 1, 2)).all()
runs_df.groupby(['task', 'image_set'])['n_trials'].describe()[['count', 'min', 'mean', 'max']]

count     min    mean     max
task        image_set                               
classic     action      10.0  1763.0  1768.7  1774.0
            greebles    10.0  1762.0  1768.3  1777.0
            taxonomic   10.0  1765.0  1768.4  1773.0
odd one out action      10.0  1799.0  1799.4  1800.0
            greebles    10.0  1800.0  1800.0  1800.0
            taxonomic   10.0  1798.0  1799.4  1800.0

Number of trials that every observer answers in draw 0: 360 each by design, fewer where the observer's sessions are shorter (timeouts; O4 in the classic task).

In [5]:
draw0 = runs_df[runs_df['draw'] == 0]
pd.DataFrame([{'task': row.task, 'image_set': row.image_set, 'trials': row.n_trials,
               **pd.Series(row.source_observer).value_counts().reindex(OBSERVERS).to_dict()}
              for row in draw0.itertuples()])

,task,image_set,trials,O1,O2,O3,O4,O5
0,classic,action,1773,360,360,360,334,359
1,classic,greebles,1765,360,360,360,328,357
2,classic,taxonomic,1769,360,360,360,331,358
3,odd one out,action,1800,360,360,360,360,360
4,odd one out,greebles,1800,360,360,360,360,360
5,odd one out,taxonomic,1799,359,360,360,360,360


In [6]:
runs_df

,observer,draw,seed,task,image_set,n_trials,responses,triplet_index,triplet_set,source_observer,source_session,raw_triplets,repetition,embedding_ready_triplets,n_triplets
0,design-B,0,0,classic,action,1773,"[1, -1, -1, 1, -1, 1, 1, -1, -1, 1, -1, 1, -1,...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[O3, O4, O4, O1, O1, O3, O4, O2, O4, O2, O4, O...","[1c, 1b, 1b, 1c, 1c, 1c, 1b, 1f, 1b, 1f, 1b, 1...","[[8, 15, 13], [2, 14, 13], [5, 1, 6], [13, 16,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[[8, 15, 13], [2, 13, 14], [5, 6, 1], [13, 16,...",1773
1,design-B,0,0,classic,greebles,1765,"[-1, -1, -1, 1, 1, 1, 1, 1, -1, -1, -1, -1, -1...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[O2, O5, O3, O2, O3, O5, O5, O4, O5, O3, O1, O...","[1f, 1c, 1a, 1f, 1a, 1c, 1c, 1f, 1c, 1a, 1a, 1...","[[0, 16, 4], [0, 3, 1], [11, 5, 13], [6, 2, 14...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[[0, 4, 16], [0, 1, 3], [11, 13, 5], [6, 2, 14...",1765
2,design-B,0,0,classic,taxonomic,1769,"[-1, 1, 1, -1, 1, -1, 1, 1, 1, -1, 1, 1, 1, -1...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[O5, O4, O4, O4, O5, O5, O4, O5, O4, O2, O3, O...","[1c, 1b, 1b, 1b, 1c, 1c, 1b, 1c, 1b, 1b, 1e, 1...","[[2, 4, 13], [0, 14, 2], [0, 7, 6], [4, 5, 13]...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[[2, 13, 4], [0, 14, 2], [0, 7, 6], [4, 13, 5]...",1769
3,design-B,0,0,odd one out,action,1800,"[1, 1, 0, 0, 1, 1, 0, 1, 0, 2, 0, 1, 0, 0, 1, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[O4, O4, O4, O3, O1, O3, O1, O4, O2, O2, O3, O...","[1a, 1a, 1a, 1d, 1d, 1d, 1d, 1a, 1a, 1a, 1d, 1...","[[15, 8, 13], [14, 2, 13], [1, 5, 6], [16, 13,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[[15, 13, 8], [14, 13, 2], [5, 6, 1], [13, 17,...",3600
4,design-B,0,0,odd one out,greebles,1800,"[1, 0, 1, 0, 2, 1, 0, 1, 1, 2, 0, 1, 0, 0, 2, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[O2, O4, O2, O1, O3, O1, O5, O2, O5, O3, O2, O...","[1e, 1d, 1e, 1b, 1d, 1b, 1f, 1e, 1f, 1d, 1e, 1...","[[16, 0, 4], [3, 0, 1], [5, 11, 13], [2, 6, 14...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[[16, 4, 0], [0, 1, 3], [5, 13, 11], [6, 14, 2...",3600
5,design-B,0,0,odd one out,taxonomic,1799,"[1, 1, 0, 0, 2, 2, 2, 2, 1, 0, 2, 1, 2, 0, 2, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[O5, O2, O5, O2, O3, O3, O1, O2, O1, O5, O2, O...","[1f, 1a, 1f, 1a, 1b, 1b, 1b, 1a, 1b, 1f, 1a, 1...","[[4, 2, 13], [14, 0, 2], [7, 0, 6], [5, 4, 13]...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[[4, 13, 2], [14, 2, 0], [0, 6, 7], [4, 13, 5]...",3598
6,design-B,1,1,classic,action,1763,"[1, -1, -1, -1, -1, 1, -1, -1, -1, 1, -1, 1, -...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[O4, O5, O1, O3, O2, O5, O5, O5, O4, O5, O5, O...","[1b, 1e, 1a, 1e, 1c, 1e, 1e, 1e, 1b, 1e, 1e, 1...","[[8, 15, 13], [2, 14, 13], [5, 1, 6], [13, 16,...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[[8, 15, 13], [2, 13, 14], [5, 6, 1], [13, 17,...",1763
7,design-B,1,1,classic,greebles,1766,"[-1, -1, -1, -1, 1, -1, -1, 1, 1, -1, -1, -1, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[O4, O4, O2, O3, O3, O3, O4, O5, O4, O3, O4, O...","[1b, 1b, 1f, 1c, 1c, 1c, 1b, 1c, 1b, 1c, 1b, 1...","[[0, 16, 4], [0, 3, 1], [11, 5, 13], [6, 2, 14...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...","[[0, 4, 16], [0, 1, 3], [11, 13, 5], [6, 14, 2...",1766
8,design-B,1,1,classic,taxonomic,1770,"[-1, 1, 1, -1, 1, -1, 1, 1, -1, -1, 1, 1, -1, ...","[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1, 1, 1, 1, 1

# Estimating dimensionality of every draw

Each draw is fitted with the significance criterion of `cblearn`. The cross-validation results of every fit are saved in `../data/triplet_accuracy_vs_dimensionality-design-B-draw{k}-{task}-{image_set}.json`. The embedding is fitted at the dimensionality of the significance criterion.

When the runs table (`RUNS_CSV`) and the json of every fit exist, the cell reads them back instead of fitting, after checking that the rebuilt pooling equals the saved one and that the table agrees with the jsons. Otherwise the fits run, and every finished fit is appended at once to `PARTIAL_CSV`, with its triplets, its dimensionality and its embedding; a rerun after an interruption skips a fit only when it has a row there and a json, and its saved triplets equal the rebuilt ones, and stops if they differ. When all fits are done, `RUNS_CSV` is written and `PARTIAL_CSV` deleted. To refit everything, delete `RUNS_CSV`, `PARTIAL_CSV` and the jsons.

In [7]:
json_names = [f"../data/triplet_accuracy_vs_dimensionality-{LABEL}-draw{d['draw']}-{d['task']}-{d['image_set']}.json"
              for _, d in runs_df.iterrows()]
loaded = os.path.exists(RUNS_CSV) and all(os.path.exists(name) for name in json_names)
keys = ['draw', 'task', 'image_set']


def dimensionality_of(name):
    """The estimated dimensionality of a fit, read from its json."""
    return read_json_dataclass(name).estimated_dimension


if loaded:
    saved = pd.read_csv(RUNS_CSV)
    assert list(saved[keys].itertuples(index=False)) == list(runs_df[keys].itertuples(index=False)), \
        'the saved runs are in another order'
    # the pooling rebuilt above must be the one that was fitted
    assert all(np.array_equal(np.array(ast.literal_eval(s)), t)
               for s, t in zip(saved['embedding_ready_triplets'], runs_df['embedding_ready_triplets'])), \
        'the rebuilt pooling differs from the saved one'
    dimensionality = [dimensionality_of(name) for name in json_names]
    assert dimensionality == list(saved['dimensionality'])
    embedding = [ast.literal_eval(s) for s in saved['embedding']]
    print(f'read {len(json_names)} saved fits and {RUNS_CSV}')
else:
    # fits finished by an earlier, interrupted run
    done = {}
    if os.path.exists(PARTIAL_CSV):
        done = {tuple(row[k] for k in keys): row for _, row in pd.read_csv(PARTIAL_CSV).iterrows()}
    dimensionality, embedding = [], []

    for (_, d), name in zip(runs_df.iterrows(), json_names):
        key = tuple(d[k] for k in keys)
        triplets = d['embedding_ready_triplets']

        if key in done and os.path.exists(name):
            row = done[key]
            if not np.array_equal(np.array(ast.literal_eval(row['embedding_ready_triplets'])), triplets):
                raise ValueError(f'{key}: the saved fit in {PARTIAL_CSV} was made on other triplets; '
                                 'delete it and the jsons of design B to refit')
            dim = dimensionality_of(name)
            assert dim == row['dimensionality']
            fitted_embedding = np.array(ast.literal_eval(row['embedding']))
            status = 'read'
        else:
            dim_result, fitted_embedding = estimate_dimensionality(triplets, random_state=SEED,
                                                                   soe_random_state=SEED)
            # saving triplet accuracy and statistics
            save_json_dataclass(dim_result, name)
            dim = dimensionality_of(name)
            # one row per finished fit, written at once so that an interrupted run keeps it
            pd.DataFrame([{**dict(zip(keys, key)), 'embedding_ready_triplets': np.asarray(triplets).tolist(),
                           'dimensionality': dim,
                           'embedding': fitted_embedding.tolist()}]).to_csv(
                PARTIAL_CSV, mode='a', header=not os.path.exists(PARTIAL_CSV), index=False)
            status = 'fitted'
        assert fitted_embedding.shape[1] == dim

        dimensionality.append(dim)
        embedding.append(fitted_embedding.tolist())

        print(f"{status}: draw {d['draw']} - {d['task']} - {d['image_set']} - {len(triplets)} triplets: "
              f"{dim} dimension(s)")

runs_df['dimensionality'] = list(dimensionality)
runs_df['embedding'] = embedding

read 60 saved fits and ../data/found-dimensionality-embeddings-design-B-draws.csv


In [8]:
if not loaded:
    # all draws; list-valued columns are saved as their repr, read back with ast.literal_eval
    to_save = runs_df.copy()
    for column in ['raw_triplets', 'responses', 'triplet_index', 'triplet_set', 'repetition', 'source_observer',
                   'source_session', 'embedding_ready_triplets', 'embedding']:
        to_save[column] = to_save[column].apply(lambda x: np.asarray(x).tolist())
    to_save.to_csv(RUNS_CSV, index=False)
    # every fit is now in RUNS_CSV
    if os.path.exists(PARTIAL_CSV):
        os.remove(PARTIAL_CSV)

# Summary

For each task and image set: the dimensionality of every observer and their mean, of pooled-5x, and the mean dimensionality over the 10 draws of design B.

In [9]:
# the individual observers and pooled-5x are read from their jsons (read_dimensionality), the source
# of the reported values
mean_b = runs_df.groupby(['task', 'image_set'])['dimensionality'].mean()

rows = []
for task in TASKS:
    for image_set in IMAGE_SETS:
        row = {'task': task, 'image_set': image_set}
        for obs in OBSERVERS:
            row[obs] = read_dimensionality(obs, task, image_set)
        row['mean O1 - O5'] = np.mean([row[obs] for obs in OBSERVERS])
        row['pooled-5x'] = read_dimensionality('pooled O1 - O5', task, image_set)
        row['design B, mean of 10 draws'] = mean_b[task, image_set]
        rows.append(row)

summary = pd.DataFrame(rows)
with pd.option_context('display.precision', 1):
    display(summary)

,task,image_set,O1,O2,O3,O4,O5,mean O1 - O5,pooled-5x,"design B, mean of 10 draws"
0,classic,action,2,1,2,2,1,1.6,4,1.0
1,classic,greebles,2,2,1,2,3,2.0,2,2.0
2,classic,taxonomic,3,2,2,1,2,2.0,2,1.4
3,odd one out,action,2,1,3,2,1,1.8,3,1.0
4,odd one out,greebles,1,2,1,2,2,1.6,2,2.0
5,odd one out,taxonomic,2,2,1,1,1,1.4,2,1.1
